**Assignment Overview**

In this assignment, you will explore two fundamental aspects of modern NLP systems: fine-tuning large language models and understanding the attention mechanism that powers them.

**Learning Goals**

By the end of this assignment, you should be able to:

* Understand when and why to fine-tune language models
* Apply QLoRA for efficient adaptation of large models
* Analyze model outputs and limitations
* Explain and implement the attention mechanism
* Interpret attention patterns and their behavior

**Important Note**

* Do not modify or delete the task structure.
* Complete each task with:
   - Clean, well-organized code
   - Relevant visualizations
   - Clear insights and explanations
* Make sure to answer all required questions.
**Submission requirements:**
* Submit a **fully executed notebook** (all cells must run and outputs should be visible).
* There is no need to attach the training and test datasets as files, but you must present them as DataFrame tables within the notebook.


## Install Required Packages and Dataset

In [1]:
%pip install --upgrade pip
%pip install -q transformers peft bitsandbytes trl
%pip install -q fastapi uvicorn openai
%pip install -q datasets tqdm
%pip install -q -U torchao

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [1]:
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    AutoModelForSeq2SeqLM,
    BitsAndBytesConfig,
    TrainingArguments,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    DataCollatorForSeq2Seq,
    pipeline,
    logging,
)
from peft import LoraConfig, TaskType, prepare_model_for_kbit_training, get_peft_model,PeftModel
from trl import SFTTrainer
import torch
import json
from datasets import Dataset, load_dataset
import pandas as pd


import warnings
warnings.filterwarnings('ignore')
logging.set_verbosity(logging.CRITICAL)

/Users/YOUR_USER/dev/cloud/nebius/lesson9_llm_training/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W0507 16:54:57.335000 22223 torch/distributed/elastic/multiprocessing/redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


## **Part 1 — QLoRA Fine-Tuning for Level-Adaptive Question Answering (75 points)**

In this part, you will fine-tune pretrained language models to answer questions at different explanation levels:

* Child — simple and intuitive explanation
* Student — clear educational explanation with moderate technical detail
* Expert — precise, technical, and domain-specific explanation

You will use a subset of the Databricks Dolly 15K dataset.

In [2]:
dataset = load_dataset("databricks/databricks-dolly-15k", split="train[:5000]")


Generating train split: 100%|██████████| 15011/15011 [00:00<00:00, 224021.95 examples/s]


In [3]:
def is_good_question(text):
    return any(q in text.lower() for q in [
        "why", "how", "what is", "explain"
    ])

filtered = [
    ex for ex in dataset
    if ex["category"] == "open_qa" and is_good_question(ex["instruction"])
]

In [4]:
len(filtered), filtered[0]

(580,
 {'instruction': 'Why can camels survive for long without water?',
  'context': '',
  'response': 'Camels use the fat in their humps to keep them filled with energy and hydration for long periods of time.',
  'category': 'open_qa'})

### **Task 1.1 — Prepare a Custom Dataset (15 points)**

Create your own instruction-tuning dataset based on a subset of databricks-dolly-15k.

For each selected question-answer pair, create versions of the answer adapted to the requested expertise level.

Example format:

    Question: What is gradient descent?
    Expertise level: child
    Answer: Gradient descent is like walking downhill step by step until you reach the lowest point.

You should prepare:

1. Training set for fine-tuning -

   Use the filtered Dolly dataset as your source of questions.

   For each question, create examples in the example format per level.

2. Test set for evaluation -
    
   Create a small test set of 20 question-answer examples.

   The test set must:

  * include examples from all three expertise levels
  * be separate from the training set
  * be manually reviewed by you for quality
  * include answers that are appropriate for the requested expertise level

  Recommendation:

  1. Save the generated dataset as a '.jsonl' file so it can be loaded and reused later.

  2. Decide on the training format according to the model architecture:
    
    - For causal language models, such as `HuggingFaceTB/SmolLM2-360M-Instruct`, use one full text field:

    ```text
    ### Question:
    ...

    ### Expertise level:
    child / student / expert

    ### Answer:
    ...```

    - For seq2seq models, such as google/flan-t5-small, separate the input and target:

    input: Question + expertise level
    target: Answer
   
  3. Before generating the full dataset, test the pipeline on a small batch of examples to verify that:

  * the LLM returns valid JSON,
  * each question receives three expertise-level answers,
  * the saved .jsonl file can be loaded correctly,
  * the format matches the training code.

In [7]:
import os
import json
import time
import random
from openai import OpenAI

# ── Configuration ─────────────────────────────────────────────────────────────
NEBIUS_API_KEY  = os.environ.get("NEBIUS_API_KEY", "your-nebius-api-key-here")
NEBIUS_BASE_URL = "https://api.studio.nebius.com/v1/"
LLM_MODEL       = os.environ.get("LLM_MODEL", "meta-llama/Llama-3.3-70B-Instruct")

LEVELS         = ["child", "student", "expert"]
TRAIN_CAUSAL   = "train_causal.jsonl"
TRAIN_SEQ2SEQ  = "train_seq2seq.jsonl"
TEST_CAUSAL    = "test_causal.jsonl"
TEST_SEQ2SEQ   = "test_seq2seq.jsonl"

random.seed(42)

# Nebius AI Studio exposes an OpenAI-compatible API
client = OpenAI(
    api_key=NEBIUS_API_KEY,
    base_url=NEBIUS_BASE_URL,
)

# ── LLM call ──────────────────────────────────────────────────────────────────
def generate_level_answers(question: str, reference: str, max_retries: int = 3) -> dict:
    """Single API call returning all three expertise-level answers as JSON."""
    prompt = (
        "You are an educational assistant.\n"
        "Given a question and a reference answer, write three explanations adapted "
        "to different expertise levels.\n\n"
        f"Question: {question}\n"
        f"Reference answer: {reference}\n\n"
        'Return ONLY a valid JSON object with exactly these three keys:\n'
        '{"child": "...", "student": "...", "expert": "..."}\n\n'
        "child   – simple, intuitive, everyday language a young child can follow.\n"
        "student – clear educational explanation with moderate technical detail.\n"
        "expert  – precise, technical, domain-specific language for a subject-matter expert."
    )
    for attempt in range(max_retries):
        try:
            resp = client.chat.completions.create(
                model=LLM_MODEL,
                messages=[{"role": "user", "content": prompt}],
                temperature=0.7,
                response_format={"type": "json_object"},
            )
            return json.loads(resp.choices[0].message.content)
        except Exception as e:
            if attempt == max_retries - 1:
                raise
            time.sleep(2 ** attempt)

# ── Formatting helpers ────────────────────────────────────────────────────────
def fmt_causal(question: str, level: str, answer: str) -> dict:
    """Single text field for decoder-only models (SmolLM2)."""
    return {
        "text": (
            f"### Question:\n{question}\n\n"
            f"### Expertise level:\n{level}\n\n"
            f"### Answer:\n{answer}"
        )
    }

def fmt_seq2seq(question: str, level: str, answer: str) -> dict:
    """Separate input/target for encoder-decoder models (flan-t5)."""
    return {
        "input":  f"Question: {question}\nExpertise level: {level}",
        "target": answer,
    }

# ── Dataset builder ───────────────────────────────────────────────────────────
def build_records(examples: list) -> tuple:
    """Return (causal_records, seq2seq_records) for a list of Dolly examples."""
    causal, seq2seq = [], []
    for i, ex in enumerate(examples):
        q, ref = ex["instruction"], ex["response"]
        answers = generate_level_answers(q, ref)
        for level in LEVELS:
            answer = answers.get(level, "").strip()
            if answer:
                causal.append(fmt_causal(q, level, answer))
                seq2seq.append(fmt_seq2seq(q, level, answer))
        if (i + 1) % 10 == 0:
            print(f"  {i + 1}/{len(examples)} questions processed")
    return causal, seq2seq

def save_jsonl(records: list, path: str) -> None:
    with open(path, "w") as f:
        for r in records:
            f.write(json.dumps(r) + "\n")

def load_jsonl(path: str) -> list:
    with open(path) as f:
        return [json.loads(line) for line in f if line.strip()]

# ─────────────────────────────────────────────────────────────────────────────
# Step 1 – Pipeline smoke-test on 3 examples
# ─────────────────────────────────────────────────────────────────────────────
print("=== Pipeline smoke-test (3 examples) ===")
smoke_causal, smoke_seq2seq = build_records(filtered[:3])

assert len(smoke_causal)  == 9, f"Expected 9 causal records, got {len(smoke_causal)}"
assert len(smoke_seq2seq) == 9, f"Expected 9 seq2seq records, got {len(smoke_seq2seq)}"

save_jsonl(smoke_causal,  "smoke_causal.jsonl")
save_jsonl(smoke_seq2seq, "smoke_seq2seq.jsonl")
assert load_jsonl("smoke_causal.jsonl")[0]["text"].startswith("### Question:")
assert "input"  in load_jsonl("smoke_seq2seq.jsonl")[0]
assert "target" in load_jsonl("smoke_seq2seq.jsonl")[0]

print("Smoke-test PASSED")
print("\nSample causal record:\n", smoke_causal[0]["text"])
print("\nSample seq2seq record:\n", smoke_seq2seq[0])

# ─────────────────────────────────────────────────────────────────────────────
# Step 2 – Train / test split
#   • First 7 shuffled questions  → test  (7 × 3 = 21 records, trimmed to 20)
#   • Remaining 573 questions     → train (573 × 3 = up to 1 719 records)
# ─────────────────────────────────────────────────────────────────────────────
shuffled = filtered.copy()
random.shuffle(shuffled)

test_src  = shuffled[:7]
train_src = shuffled[7:]

print(f"\nTrain source questions : {len(train_src)}")
print(f"Test  source questions : {len(test_src)}")

# ─────────────────────────────────────────────────────────────────────────────
# Step 3 – Generate training dataset
# ─────────────────────────────────────────────────────────────────────────────
print("\n=== Generating training dataset ===")
train_causal_recs, train_seq2seq_recs = build_records(train_src)

save_jsonl(train_causal_recs,  TRAIN_CAUSAL)
save_jsonl(train_seq2seq_recs, TRAIN_SEQ2SEQ)
print(f"Saved {len(train_causal_recs)} causal records  → {TRAIN_CAUSAL}")
print(f"Saved {len(train_seq2seq_recs)} seq2seq records → {TRAIN_SEQ2SEQ}")

# ─────────────────────────────────────────────────────────────────────────────
# Step 4 – Generate test dataset (exactly 20 examples, all three levels present)
# ─────────────────────────────────────────────────────────────────────────────
print("\n=== Generating test dataset ===")
test_causal_recs, test_seq2seq_recs = build_records(test_src)

test_causal_recs  = test_causal_recs[:20]
test_seq2seq_recs = test_seq2seq_recs[:20]

save_jsonl(test_causal_recs,  TEST_CAUSAL)
save_jsonl(test_seq2seq_recs, TEST_SEQ2SEQ)
print(f"Saved {len(test_causal_recs)} causal records  → {TEST_CAUSAL}")
print(f"Saved {len(test_seq2seq_recs)} seq2seq records → {TEST_SEQ2SEQ}")

# ─────────────────────────────────────────────────────────────────────────────
# Step 5 – Verify loading and display as DataFrames
# ─────────────────────────────────────────────────────────────────────────────
train_causal_df  = pd.DataFrame(load_jsonl(TRAIN_CAUSAL))
train_seq2seq_df = pd.DataFrame(load_jsonl(TRAIN_SEQ2SEQ))
test_causal_df   = pd.DataFrame(load_jsonl(TEST_CAUSAL))
test_seq2seq_df  = pd.DataFrame(load_jsonl(TEST_SEQ2SEQ))

print("\n=== Dataset Summary ===")
print(f"Train causal  : {len(train_causal_df)} rows | columns: {list(train_causal_df.columns)}")
print(f"Train seq2seq : {len(train_seq2seq_df)} rows | columns: {list(train_seq2seq_df.columns)}")
print(f"Test  causal  : {len(test_causal_df)} rows")
print(f"Test  seq2seq : {len(test_seq2seq_df)} rows")

print("\n--- Training set (causal, first 5 rows) ---")
display(train_causal_df.head())

print("\n--- Training set (seq2seq, first 5 rows) ---")
display(train_seq2seq_df.head())

print("\n--- Test set — causal (all 20 rows) ---")
display(test_causal_df)

print("\n--- Test set — seq2seq (all 20 rows) ---")
display(test_seq2seq_df)


=== Pipeline smoke-test (3 examples) ===
Smoke-test PASSED

Sample causal record:
 ### Question:
Why can camels survive for long without water?

### Expertise level:
child

### Answer:
Camels can go a long time without water because they have a special hump on their back that stores fat, which gives them energy and helps keep them hydrated until they find water again.

Sample seq2seq record:
 {'input': 'Question: Why can camels survive for long without water?\nExpertise level: child', 'target': 'Camels can go a long time without water because they have a special hump on their back that stores fat, which gives them energy and helps keep them hydrated until they find water again.'}

Train source questions : 573
Test  source questions : 7

=== Generating training dataset ===
  10/573 questions processed
  20/573 questions processed
  30/573 questions processed
  40/573 questions processed
  50/573 questions processed
  60/573 questions processed
  70/573 questions processed
  80/573 quest

,text
0,### Question:\nWhat is Tableau?\n\n### Experti...
1,### Question:\nWhat is Tableau?\n\n### Experti...
2,### Question:\nWhat is Tableau?\n\n### Experti...
3,### Question:\nWhy is sailing so much fun?\n\n...
4,### Question:\nWhy is sailing so much fun?\n\n...



--- Training set (seq2seq, first 5 rows) ---


,input,target
0,Question: What is Tableau?\nExpertise level: c...,Tableau is a company that helps people make se...
1,Question: What is Tableau?\nExpertise level: s...,Tableau is a data visualization software that ...
2,Question: What is Tableau?\nExpertise level: e...,Tableau is a cutting-edge data visualization p...
3,Question: Why is sailing so much fun?\nExperti...,Sailing is fun because you get to go outside a...
4,Question: Why is sailing so much fun?\nExperti...,Sailing is an enjoyable activity that combines...



--- Test set — causal (all 20 rows) ---


,text
0,### Question:\nHow many feet are in 1 mile?\n\...
1,### Question:\nHow many feet are in 1 mile?\n\...
2,### Question:\nHow many feet are in 1 mile?\n\...
3,### Question:\nWhat is Project Mosaic?\n\n### ...
4,### Question:\nWhat is Project Mosaic?\n\n### ...
5,### Question:\nWhat is Project Mosaic?\n\n### ...
6,### Question:\nHow many different Alaska Nativ...
7,### Question:\nHow many different Alaska Nativ...
8,### Question:\nHow many different Alaska Nativ...
9,### Question:\nWhy do people eat food?\n\n### ...



--- Test set — seq2seq (all 20 rows) ---


,input,target
0,Question: How many feet are in 1 mile?\nExpert...,A mile is a really long distance. It's like dr...
1,Question: How many feet are in 1 mile?\nExpert...,"To understand how many feet are in a mile, we ..."
2,Question: How many feet are in 1 mile?\nExpert...,The conversion of miles to feet is based on th...
3,Question: What is Project Mosaic?\nExpertise l...,Project Mosaic is a tool that helps computers ...
4,Question: What is Project Mosaic?\nExpertise l...,Project Mosaic is an extension of the Apache S...
5,Question: What is Project Mosaic?\nExpertise l...,Project Mosaic is a scalable geospatial proces...
6,Question: How many different Alaska Native tri...,There are 228 different groups of Alaska Nativ...
7,Question: How many different Alaska Native tri...,Alaska is home to a diverse range of indigenou...
8,Question: How many different Alaska Native tri...,The State of Alaska is comprised of 228 federa...
9,Question: Why do people eat food?\nExpertise l...,People eat food because it gives us energy to ...


### **Task 1.2 — Fine-Tune Models with QLoRA (10 points)**

Fine-tune the models using QLoRA, meaning:

1. Load the base model in 4-bit quantization
2. Add LoRA adapters
3. Train only the adapter parameters

You should repeat the fine-tuning process for both models:

1. HuggingFaceTB/SmolLM2-360M-Instruct
2. google/flan-t5-small

Pay attention each model requires the dataset to be formated diffrently.

### **Task 1.3 — Evaluation (20 points)**

Evaluate both fine-tuned models on your 20-example test set.

For each model, compare:

* outputs before fine-tuning
* outputs after fine-tuning
* whether the answer matches the requested expertise level
* whether the answer is clear and relevant
* whether the answer stays faithful to the question

Evaluate using this table:

|Question|Level|Base Output|Fine-tuned Output|Expected Answer|Better after FT?|Notes|
|-------|-------|-------|-------|-------|-------|-------|


### **Task 1.4 — Model Comparison and Discussion (15 points)**

Compare the performance of the two models:

* Which model adapted better to the expertise levels?
* Which model produced clearer answers?
* Which model followed the requested format better?
* Did one model hallucinate more than the other?

Explain any differences you observe.

In your discussion, consider that:

* SmolLM2-360M-Instruct is a decoder-only instruction model
* flan-t5-small is an encoder-decoder instruction model
* Different architectures may behave differently on instruction-following and text generation tasks

### **Task 1.5 - Conceptual Questions (15 points)**

Answer the following questions:

1. What changed after fine-tuning?
   
   Discuss whether the model became better at adapting its answer to the requested expertise level.
2. Why is QLoRA more memory efficient?
   
   Explain the role of 4-bit quantization and LoRA adapters.
3. What happens if you increase the LoRA rank?
   
   Discuss the tradeoff between model capacity, memory usage, and overfitting risk.
4. Why use LoRA / QLoRA instead of prompt engineering?

      Discuss:

      * In what cases prompt engineering is sufficient
      * When fine-tuning becomes necessary
      * What advantages QLoRA provides over prompting
      * What are the trade-offs (cost, flexibility, control)

## **Part 2 — Understanding Attention (25 points)**

Goal:

Build intuition for how attention works.

Task:

You will implement a simple attention mechanism from scratch (PyTorch) and visualize its behavior.

### **Task 2.1 Implement Scaled Dot-Product Attention (5 points)**

Given:

* Query (Q)
* Key (K)
* Value (V)

Compute:

$$ Attention(Q,K,V) = softmax(\frac{QK^T}{\sqrt{d_k}})V$$

In [ ]:
import torch
import torch.nn.functional as F
import math

def scaled_dot_product_attention(Q, K, V, mask=None):
    """
    Implements Scaled Dot-Product Attention.

    Args:
        Q: (batch, heads, seq_len_q, d_k)
        K: (batch, heads, seq_len_k, d_k)
        V: (batch, heads, seq_len_v, d_v)
        mask: (batch, heads, seq_len_q, seq_len_k) or None

    Returns:
        output: (batch, heads, seq_len_q, d_v)
        attention_weights: (batch, heads, seq_len_q, seq_len_k)
    """


    #  Get dimension for scaling
    d_k = Q.size(-1)


    # Compute attention scores

    scores = # TODO: compute dot-product between Q and K^T


    # Scale the scores

    # TODO: divide scores by sqrt(d_k)


    #  Apply mask (if given)

    if mask is not None:
        # TODO: mask out invalid positions (set to -inf)
        pass


    # Softmax to get attention

    attention_weights = # TODO: apply softmax over last dimension


    # Compute weighted sum

    output = #TODO: multiply attention_weights with V

    return output, attention_weights

### **Task 2.2 Visualize Attention (5 points)**


Plot attention weights as a heatmap for the given sentence


In [ ]:
import torch
import matplotlib.pyplot as plt
import seaborn as sns
import math



sentence = "the cat sat on the mat"


tokens = # TODO: split sentence into tokens

seq_len = # TODO: compute sequence length



# Create dummy embeddings

d_model = 16


embeddings = # TODO: initialize random embeddings of shape (seq_len, d_model)

# Treat embeddings as Q, K, V
Q = embeddings
K = embeddings
V = embeddings



# Compute Scaled Dot-Product Attention

attention_weights = # TODO: compute attention weights (use function from above)



# Plot attention heatmap

plt.figure(figsize=(8, 6))

# TODO: plot heatmap using seaborn
# - use attention_weights
# - set xticklabels and yticklabels to tokens
# - choose a colormap (e.g., "viridis")

plt.title("Attention Weights Heatmap")

# TODO: label axes
# plt.xlabel(...)
# plt.ylabel(...)

# TODO: rotate ticks if needed

plt.show()



### **Task 2.3 Experiments (15 points)**

**Experiment 1 — Change One Word:**

1. Use a simple sentence, for example:
    the cat sat on the mat
2. Compute and plot the attention weights as a heatmap.
3. Change one word in the sentence, for example:
    the dog sat on the mat
4. Recompute and plot the attention heatmap.
5. Compare the two heatmaps and explain whether the attention pattern changed.

**Experiment 2 — Compare Different Attention Heads:**

Repeat the attention visualization using at least two different attention heads.

For each head, create separate projection matrices:

$$W_Q, W_K, W_V$$

Use them to compute:
$$Q=XW_Q, K=XW_K, V=XW_V$$

Then compute and plot the attention weights for each head.

**Questions**

Answer briefly:

1. Did changing one word affect the attention weights? Why or why not?
2. Do different attention heads focus on different tokens?
3. Why might multi-head attention be useful in Transformer models?